# 00 — Setup R-GFM (official code)

Clones `USTC-DataDarknessLab/R-GFM` — the official implementation of *Learning Graph Foundation Models on Riemannian Graph-of-Graphs* — and prepares a Colab GPU runtime to run it.

**Before running:** Runtime -> Change runtime type -> GPU (A100 if you have Colab Pro/Pro+; T4 works for the small/medium datasets below but will be slower and can't run the paper's large-scale ogbn-Arxiv/Reddit setting).

**Known repo quirks (checked against the `main` branch):**
- The repo's README references a `requirements.txt` that **does not actually exist** in the repo. This notebook installs the dependencies inferred from the actual imports instead (`torch`, `torch_geometric`, `torch_scatter`/`torch_sparse`, `geoopt`, `ogb`, `scikit-learn`).
- `graph_aug/cuda_backend.py` is orphaned dead code — it JIT-compiles from `cuda_kernels/*_host.cpp` files that don't exist in the repo. Don't import it. The real runtime path (used by `utils/data/augmentation.py`) is the precompiled `graph_aug/graph_aug_cuda*.so` built via `setup.py build_ext --inplace`, exactly as the README instructs — that path **is** consistent (the `.cu`/`.cpp` sources it lists all exist and match the pybind11 bindings).

In [ ]:
# 1. GPU check.
!nvidia-smi | head -n 15

In [ ]:
# 2. Clone the official repository.
import os
REPO_DIR = '/content/R-GFM'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/USTC-DataDarknessLab/R-GFM.git $REPO_DIR
else:
    print(f'{REPO_DIR} already exists — pulling latest')
    !git -C $REPO_DIR pull --ff-only

In [ ]:
# 3. Show what's inside.
!ls -la $REPO_DIR
!ls -la $REPO_DIR/graph_aug

In [ ]:
# 4. CUDA toolkit check — Colab images ship nvcc matching the driver's CUDA runtime.
!nvcc --version

In [ ]:
# 5. Install torch first (pinned per the paper repo's stated requirement, torch==2.8.0),
# then detect the exact torch+cuda build string so the PyG wheel index matches it exactly.
!pip install --quiet torch==2.8.0

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
print('torch', torch.__version__, '-> wheel tag', TORCH_VER, CUDA_TAG, '| cuda available:', torch.cuda.is_available())

In [ ]:
# 6. torch_geometric + torch_scatter/torch_sparse matched to the detected build.
import os
wheel_url = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
print('PyG wheel index:', wheel_url)
!pip install --quiet torch_geometric
os.system(f'pip install --quiet torch_scatter torch_sparse -f {wheel_url}')

In [ ]:
# 7. Remaining dependencies (geoopt for Riemannian manifolds/optimizers, ogb for ogbl-collab,
# scikit-learn for AUC-ROC). Not pinned upstream (no requirements.txt exists in the repo),
# so we take current releases.
!pip install --quiet geoopt ogb scikit-learn

In [ ]:
# 8. Build the custom CUDA graph-augmentation extension. This is required — main.py
# will crash on import otherwise (utils/data/augmentation.py imports graph_aug.graph_aug_cuda).
%cd $REPO_DIR/graph_aug
!python setup.py build_ext --inplace
%cd $REPO_DIR

In [ ]:
# 9. Mount Drive for persistent datasets/checkpoints/results across session resets.
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/R-GFM'
DATA_DIR = f'{BASE}/datasets'
CKPT_DIR = f'{BASE}/checkpoints'
RESULTS_DIR = f'{BASE}/results'
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Drive workspace:', BASE)

In [ ]:
# 10. Sanity check — imports, versions, and the built extension all resolve.
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch, torch_geometric, geoopt, ogb
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
print('pyg', torch_geometric.__version__)
print('geoopt', geoopt.__version__)
print('ogb', ogb.__version__)

import graph_aug.graph_aug_cuda as graph_aug_cuda
print('graph_aug_cuda loaded OK:', [n for n in dir(graph_aug_cuda) if not n.startswith('_')])

**Done.** Move on to `01_node_classification.ipynb`.

Every training notebook in this folder expects these Colab globals to already be set from this notebook: `REPO_DIR`, `DATA_DIR`, `CKPT_DIR`, `RESULTS_DIR`. If you start a fresh runtime, re-run this notebook first (Drive mount + venv install are the slow parts; git clone and the CUDA build are fast).